# Project 3 — SQL Data Analysis
### DecodeLabs | Data Analytics Internship — Batch 2026

**Analyst:** Intern, DecodeLabs Data Analytics Track
**Dataset:** `ecommerce_orders_cleaned.csv` (verified output of Project 1) loaded into a SQLite database
**Objective:** Bridge the gap between a massive dataset and specific
business answers through pure relational logic — writing `SELECT` queries
with `WHERE`, `GROUP BY`, `HAVING`, and aggregate functions (`COUNT`, `SUM`,
`AVG`) to extract actionable insight.

This notebook follows the **query execution order**, not the syntax order —
the database evaluates `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT`
→ `ORDER BY`, regardless of the order these clauses are typed in.

All queries here are also saved as a standalone, reviewable file at
[`sql/queries.sql`](../sql/queries.sql).


## 1. Setup & Database Connection

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

DATA_PATH = Path('../data/ecommerce_orders_cleaned.csv')
DB_PATH = Path('../database/ecommerce.db')

print("Environment ready.")


Environment ready.


## 2. Build the Database
Loading the verified, cleaned CSV from Project 1 into a SQLite database —
turning a flat file into a queryable relational table.


In [2]:
df = pd.read_csv(DATA_PATH)

conn = sqlite3.connect(DB_PATH)
df.to_sql('orders', conn, if_exists='replace', index=False)

print(f"Database built at {DB_PATH.resolve().name}")
print(f"Table 'orders' loaded with {len(df):,} rows.")


Database built at ecommerce.db
Table 'orders' loaded with 1,200 rows.


## 3. Basic SELECT — Sanity Check

In [3]:
query = """
SELECT COUNT(*) AS total_orders
FROM orders;
"""

pd.read_sql(query, conn)


,total_orders
0,1200


## 4. WHERE — Filtering High-Value Orders
The `WHERE` clause acts as a row-level funnel: isolating orders above $3,000 before any aggregation happens.

In [4]:
query = """SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice
FROM orders
WHERE TotalPrice > 3000
ORDER BY TotalPrice DESC;"""

result = pd.read_sql(query, conn)
result

,OrderID,Product,Quantity,UnitPrice,TotalPrice
0,ORD200789,Tablet,5,691.28,3456.40
1,ORD201122,Monitor,5,678.19,3390.95
2,ORD200632,Laptop,5,678.16,3390.80
3,ORD200469,Chair,5,676.98,3384.90
4,ORD200328,Tablet,5,674.04,3370.20
5,ORD200107,Printer,5,670.75,3353.75
6,ORD200326,Laptop,5,670.48,3352.40
7,ORD201065,Printer,5,666.80,3334.00
8,ORD201031,Phone,5,664.51,3322.55
9,ORD200463,Laptop,5,662.78,3313.90


**Finding:** 34 orders exceed $3,000 — all are 5-unit orders of premium-priced items (Tablet, Laptop, Monitor, Chair, Printer, Phone). These are the same high-value outliers flagged in Project 2's IQR analysis — confirmed here independently via SQL.

## 5. GROUP BY + Aggregation — Revenue and Order Count per Product

In [5]:
query = """SELECT
    Product,
    COUNT(*) AS order_count,
    SUM(TotalPrice) AS total_revenue,
    ROUND(AVG(TotalPrice), 2) AS avg_order_value
FROM orders
GROUP BY Product
ORDER BY total_revenue DESC;"""

result = pd.read_sql(query, conn)
result

,Product,order_count,total_revenue,avg_order_value
0,Chair,178,195620.11,1098.99
1,Printer,181,195612.61,1080.73
2,Laptop,173,192126.56,1110.56
3,Tablet,179,186568.95,1042.28
4,Monitor,163,175651.41,1077.62
5,Desk,170,167459.93,985.06
6,Phone,156,151722.39,972.58


**Finding:** Chair ($195,620) and Printer ($195,613) are effectively tied for the top revenue-generating product — confirming the Project 2 EDA finding via direct SQL aggregation.

## 6. GROUP BY — Order Status Distribution (with Percentage Share)
Using a scalar subquery in the `SELECT` list to compute each status's share of the total.

In [6]:
query = """SELECT
    OrderStatus,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS pct_of_total
FROM orders
GROUP BY OrderStatus
ORDER BY order_count DESC;"""

result = pd.read_sql(query, conn)
result

,OrderStatus,order_count,pct_of_total
0,Cancelled,250,20.83
1,Returned,247,20.58
2,Pending,237,19.75
3,Shipped,235,19.58
4,Delivered,231,19.25


**Finding:** Cancelled (20.83%) and Returned (20.58%) together account for **41.4%** of all orders — verifying the fulfillment-risk finding from Project 2 directly against the database.

## 7. GROUP BY — Average Order Value (AOV) by Payment Method

In [7]:
query = """SELECT
    PaymentMethod,
    COUNT(*) AS order_count,
    ROUND(AVG(TotalPrice), 2) AS avg_order_value
FROM orders
GROUP BY PaymentMethod
ORDER BY avg_order_value DESC;"""

result = pd.read_sql(query, conn)
result

,PaymentMethod,order_count,avg_order_value
0,Credit Card,234,1127.55
1,Gift Card,230,1070.97
2,Cash,246,1056.04
3,Online,258,1017.22
4,Debit Card,232,1001.56


**Finding:** Credit Card orders have the highest AOV (\$1,127.55); Debit Card the lowest (\$1,001.56).

## 8. GROUP BY — Revenue by Marketing / Referral Channel

In [8]:
query = """SELECT
    ReferralSource,
    COUNT(*) AS order_count,
    ROUND(SUM(TotalPrice), 2) AS total_revenue,
    ROUND(AVG(TotalPrice), 2) AS avg_order_value
FROM orders
GROUP BY ReferralSource
ORDER BY total_revenue DESC;"""

result = pd.read_sql(query, conn)
result

,ReferralSource,order_count,total_revenue,avg_order_value
0,Instagram,259,275285.45,1062.88
1,Email,250,261808.55,1047.23
2,Google,241,250441.48,1039.18
3,Facebook,228,250410.90,1098.29
4,Referral,222,226815.58,1021.69


**Finding:** Instagram (\$275,285) and Email (\$261,809) are the top two revenue-generating channels — consistent with Project 2.

## 9. HAVING — Repeat Customers
`HAVING` filters **after** aggregation — the opposite of `WHERE`, which
filters individual rows before grouping. Here it isolates customers who
placed more than one order.


In [9]:
query = """
SELECT
    CustomerID,
    COUNT(*) AS order_count,
    ROUND(SUM(TotalPrice), 2) AS total_spent
FROM orders
GROUP BY CustomerID
HAVING COUNT(*) > 1
ORDER BY total_spent DESC
LIMIT 10;
"""

pd.read_sql(query, conn)


,CustomerID,order_count,total_spent
0,C38840,2,5723.23
1,C97593,2,2855.22
2,C98474,2,2175.34
3,C70659,2,1853.96
4,C94569,2,1675.35
5,C46651,2,1360.89
6,C35852,2,1248.39
7,C14847,2,1097.81
8,C91155,2,704.82
9,C21191,2,647.77


In [10]:
query = """
SELECT COUNT(*) AS repeat_customer_count
FROM (
    SELECT CustomerID
    FROM orders
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
);
"""

pd.read_sql(query, conn)


,repeat_customer_count
0,11


**Finding:** Only 11 of 1,189 unique customers placed more than one order (~0.9% repeat rate) — a strong signal for a customer-retention initiative, since acquisition currently dominates the order book almost entirely.

## 10. Date Functions + GROUP BY — Yearly Revenue Trend

In [11]:
query = """SELECT
    strftime('%Y', Date) AS year,
    COUNT(*) AS order_count,
    ROUND(SUM(TotalPrice), 2) AS total_revenue
FROM orders
GROUP BY year
ORDER BY year;"""

result = pd.read_sql(query, conn)
result

,year,order_count,total_revenue
0,2023,510,552643.24
1,2024,459,480235.87
2,2025,231,231882.85


**Finding:** Revenue declines year-over-year in this sample (2023: \$552,643 -> 2024: \$480,236 -> 2025 partial-year: \$231,883). 2025 is a partial year (data through June), so this is not a like-for-like comparison — it should not be read as a definitive downward trend without annualizing 2025.

## 11. Subquery in WHERE — At-Risk, Above-Average-Value Orders
Combining a multi-condition `WHERE` with a subquery: orders that are Cancelled/Returned **and** above the dataset's average order value.

In [12]:
query = """SELECT OrderID, Product, OrderStatus, TotalPrice
FROM orders
WHERE OrderStatus IN ('Cancelled', 'Returned')
  AND TotalPrice > (SELECT AVG(TotalPrice) FROM orders)
ORDER BY TotalPrice DESC
LIMIT 10;"""

result = pd.read_sql(query, conn)
result

,OrderID,Product,OrderStatus,TotalPrice
0,ORD201122,Monitor,Returned,3390.95
1,ORD200469,Chair,Cancelled,3384.90
2,ORD200328,Tablet,Cancelled,3370.20
3,ORD200326,Laptop,Returned,3352.40
4,ORD200527,Chair,Cancelled,3267.35
5,ORD200768,Tablet,Cancelled,3267.30
6,ORD200889,Monitor,Cancelled,3253.60
7,ORD200802,Chair,Cancelled,3223.20
8,ORD200957,Monitor,Returned,3219.45
9,ORD200086,Printer,Cancelled,3215.15


**Finding:** High-value orders are also getting cancelled/returned — this is the costliest slice of the fulfillment-risk problem and the first place a root-cause investigation should look.

## 12. GROUP BY — Coupon Usage and Its Effect on Order Value

In [13]:
query = """SELECT
    CouponCode,
    COUNT(*) AS order_count,
    ROUND(AVG(TotalPrice), 2) AS avg_order_value
FROM orders
GROUP BY CouponCode
ORDER BY order_count DESC;"""

result = pd.read_sql(query, conn)
result

,CouponCode,order_count,avg_order_value
0,Freeship,313,1070.41
1,Nocoupon,309,1043.37
2,Winter15,292,1035.90
3,Save10,286,1065.87


**Finding:** Orders with no coupon (`Nocoupon`) have a lower average order value (\$1,043.37) than `Freeship` (\$1,070.41) or `Save10` (\$1,065.87) orders — suggesting coupon usage correlates with, but does not necessarily cause, larger baskets. This is a correlation, not a causal claim.

## 13. Close the Database Connection

In [14]:
conn.close()
print("Connection closed.")


Connection closed.


## 14. Executive Summary

| # | SQL Technique | Finding | Business Implication |
|---|---|---|---|
| 1 | `WHERE` + `ORDER BY` | 34 orders exceed \$3,000, all 5-unit bulk orders | Candidate list for account-manager follow-up / bulk-order review |
| 2 | `GROUP BY` + `SUM`/`AVG` | Chair and Printer are the top two revenue products (near tie) | Prioritize inventory & supplier reliability for these two lines |
| 3 | `GROUP BY` + scalar subquery | 41.4% of orders are Cancelled or Returned | Confirms the Project 2 fulfillment-risk finding independently via SQL |
| 4 | `GROUP BY` | Credit Card has the highest AOV, Debit Card the lowest | Investigate checkout friction on lower-AOV payment methods |
| 5 | `HAVING` | Only ~0.9% of customers are repeat buyers | Retention program is a high-leverage opportunity — acquisition currently dominates |
| 6 | Date functions + `GROUP BY` | Revenue trend across 2023-2025 (2025 partial year) | Do not treat the partial 2025 total as a like-for-like annual comparison |
| 7 | Subquery in `WHERE` | High-value orders are disproportionately represented in Cancelled/Returned | Highest-value root-cause investigation target |
| 8 | `GROUP BY` | Coupon-using orders have slightly higher AOV than no-coupon orders | Correlational signal only — not evidence that coupons *cause* larger baskets |

## 15. Conclusion

This project demonstrates the full SQL toolkit required to turn a raw table
into decision-ready answers: `SELECT`, `WHERE`, `GROUP BY`, `HAVING`,
aggregate functions, scalar subqueries, and date functions — all executed
against a real relational database built from the Project 1 output. Every
major finding from Project 2's Python-based EDA was independently
cross-verified here using pure SQL, and one new finding (the ~0.9% repeat
customer rate) emerged that wasn't surfaced in the earlier analysis.

Full findings are also documented in
[`reports/sql_insights_summary.md`](../reports/sql_insights_summary.md).

---
**DecodeLabs | Data Analytics Internship — Project 3 of the Industrial Training Kit**
